# smart-commit: mining preference pairs on Kaggle

1. Settings: **Accelerator: GPU T4 x2**, **Internet: on** (needs a verified phone).
2. **Add Input** → the private dataset made by `training/kaggle_mine_bundle.py`.
3. **Save Version → Save & Run All (Commit)**: it runs in the background, up to 12 h.
4. Download `/kaggle/working/<job>.tar.gz` from the version's **Output**, `cat` the
   `candidates.jsonl` of a model's shards back in the bundle's order (shard K holds
   every Nth input from K) and continue with `sc-training pairs` locally.

llama.cpp's CUDA build and the models are downloaded and checked against the SHA-256
in `runs.json`. Jobs go one per GPU, the longest first; each is packed as soon as it ends.

In [ ]:
import glob, hashlib, json, os, pathlib, subprocess, tarfile, time, urllib.request
bundle = pathlib.Path(glob.glob('/kaggle/input/**/runs.json', recursive=True)[0]).parent
spec = json.loads((bundle / 'runs.json').read_text())
print(bundle, json.dumps(spec['jobs'], indent=1))
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(subprocess.run(['ldd', '--version'], capture_output=True, text=True).stdout.splitlines()[0])


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

llama = spec['llama']
llama_dir = pathlib.Path('/kaggle/temp/llama')
llama_dir.mkdir(parents=True, exist_ok=True)
for asset in llama['assets']:
    target = llama_dir / asset['name']
    if not target.exists():
        urllib.request.urlretrieve(llama['url'].format(release=llama['release'], name=asset['name']), target)
    assert sha256(target) == asset['sha256'], asset['name']
    with tarfile.open(target) as archive:
        archive.extractall(llama_dir, filter='data')
server_dir = llama_dir / f"llama-{llama['release']}"
server = server_dir / 'llama-server'
libs = ':'.join([str(server_dir), *map(str, llama_dir.glob('cudart-*')), os.environ.get('LD_LIBRARY_PATH', '')])
def job_env(gpu):
    return {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu), 'LD_LIBRARY_PATH': libs}
listing = subprocess.run([str(server), '--list-devices'], capture_output=True, text=True, env=job_env('0,1'))
print(listing.stdout, listing.stderr[-1500:])
assert 'CUDA0' in listing.stdout, 'llama-server sees no CUDA device: the build does not fit this image'


In [ ]:
from huggingface_hub import hf_hub_download
ggufs = {}
for key, m in spec['models'].items():
    path = hf_hub_download(m['repo'], m['file'], revision=m['revision'], local_dir=f'/kaggle/temp/{key}')
    assert os.path.getsize(path) == m['size'] and sha256(path) == m['sha256'], key
    ggufs[key] = path
print(ggufs)


In [ ]:
sampling = spec['sampling']
def command(job, shard, out, samples):
    return ['python', '-I', str(bundle / 'mine_pairs.py'), str(shard), str(out), ggufs[job['model']],
            '--prompts', str(shard / 'prompts.jsonl'), '--server', str(server), '--device', 'CUDA0',
            '--rules', sampling['rules'], '--diff', sampling['diff'], '--samples', str(samples),
            '--temperature', str(sampling['temperature']), '--top-p', str(sampling['top_p']),
            '--seed', str(sampling['seed']), '--workers', str(sampling['workers'])]

# Four inputs of each model, to fail now rather than after an hour.
for model in spec['models']:
    job = next(j for j in spec['jobs'] if j['model'] == model)
    tiny = pathlib.Path(f'/kaggle/temp/smoke-{model}-shard')
    tiny.mkdir(parents=True, exist_ok=True)
    for name in ('inputs', 'meta', 'prompts'):
        lines = (bundle / job['shard'] / f'{name}.jsonl').read_text().splitlines()[:4]
        (tiny / f'{name}.jsonl').write_text(''.join(line + '\n' for line in lines))
    out = pathlib.Path(f'/kaggle/temp/smoke-{model}')
    done = subprocess.run(command(job, tiny, out, 2), capture_output=True, text=True, env=job_env(0))
    print(model, done.returncode, done.stdout[-300:], done.stderr[-600:])
    rows = [json.loads(line) for line in (out / 'candidates.jsonl').read_text().splitlines()]
    assert len(rows) == 4 and all(r['replies'] and all(r['replies']) for r in rows), model
    print(rows[0]['replies'])


In [ ]:
work = pathlib.Path('/kaggle/working')
gpus = len([l for l in subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.splitlines() if l.startswith('GPU')])
pending, running, attempts = list(spec['jobs']), [], {}
while pending or running:
    busy = {gpu for gpu, _, _ in running}
    for gpu in range(gpus):
        if pending and gpu not in busy:
            job = pending.pop(0)
            attempts[job['name']] = attempts.get(job['name'], 0) + 1
            shard = bundle / job['shard']
            assert sha256(shard / 'prompts.jsonl') == job['prompts_sha256'], job['name']
            out = work / 'mined' / job['name']
            out.mkdir(parents=True, exist_ok=True)
            cmd = command(job, shard, out, sampling['samples'])
            print(f"gpu {gpu}: {' '.join(cmd)}", flush=True)
            log = open(work / f"{job['name']}.log", 'a')
            proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=job_env(gpu))
            running.append((gpu, job, proc))
            busy.add(gpu)
    time.sleep(30)
    for item in list(running):
        gpu, job, proc = item
        if proc.poll() is None:
            continue
        running.remove(item)
        name = job['name']
        candidates = work / 'mined' / name / 'candidates.jsonl'
        done = len(candidates.read_text().splitlines()) if candidates.exists() else 0
        print(f"{name}: exit {proc.returncode}, {done}/{job['inputs']} inputs", flush=True)
        print(open(work / f'{name}.log').read()[-600:], flush=True)
        if proc.returncode == 0 and done == job['inputs']:
            # Packed right away, so a session that dies later keeps it.
            subprocess.run(['tar', 'czf', str(work / f'{name}.tar.gz'), '-C', str(work / 'mined'),
                            f'{name}/candidates.jsonl', f'{name}/meta.jsonl', '-C', str(work), f'{name}.log'], check=True)
        elif attempts[name] < 2:
            pending.insert(0, job)  # resumes from the inputs it finished
        else:
            print(f'{name} FAILED twice: its candidates stay in {candidates}', flush=True)


In [ ]:
# The shards stay in mined/ only until the archives are made.
!rm -rf {work}/mined/*/inputs.jsonl {work}/mined/*/prompts.jsonl
!ls -la {work}
